# Quick feature visualization
Loads `artifacts/text_features.csv` and plots simple relationships between age and features.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


In [ ]:
from pathlib import Path

# Prefer batch1 if present
candidates = [
    Path('artifacts/text_features_batch1.csv'),
    Path('../artifacts/text_features_batch1.csv'),
    Path('artifacts/text_features.csv'),
    Path('../artifacts/text_features.csv'),
]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError('Could not find artifacts/text_features*.csv (run analysis first)')
df = pd.read_csv(path, low_memory=False)
print(df.shape)
df.head(2)

## Clean + basic filters

In [ ]:
# Keep rows with an age midpoint
d = df.copy()
d['age_mid'] = pd.to_numeric(d.get('age_mid'), errors='coerce')
d = d[d['age_mid'].notna()].copy()

# Numeric amazon fields (if present)
for c in ['Num Reviews','Average Rating','Price','yt_view_count','yt_like_count','yt_duration']:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors='coerce')

print('rows w/ age_mid:', len(d))
d[['age_mid','num_words','mean_sentence_length_words','mean_word_length_chars']].describe()

## Scatter: age vs basic text features

In [ ]:
features = [
    'num_words',
    'num_sentences',
    'mean_sentence_length_words',
    'mean_word_length_chars',
    'num_pronouns',
    'ttr',
    'mean_log10_corpus_freq',
    'rare_frac_corpus_le_5',
    'flesch_kincaid_grade_est',
    'avg_syllables_per_word',
    'concept_n_events_parsed',
    'concept_mean_event_words',
]
features = [f for f in features if f in d.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax, f in zip(axes, features):
    sns.regplot(data=d, x='age_mid', y=f, ax=ax, scatter_kws={'s':8, 'alpha':0.25}, line_kws={'color':'black'})
    ax.set_title(f)
plt.tight_layout()


## Boxplots by coarse age bins

In [ ]:
bins = [0, 3, 5, 7, 9, 12, 16]
labels = [f'{bins[i]}-{bins[i+1]}' for i in range(len(bins)-1)]
d['age_bin'] = pd.cut(d['age_mid'], bins=bins, labels=labels, include_lowest=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for ax, f in zip(axes, features[:4]):
    sns.boxplot(data=d, x='age_bin', y=f, ax=ax)
    ax.set_title(f)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()


## Amazon popularity/ratings (if present)

In [ ]:
if 'Num Reviews' in d.columns:
    plt.figure(figsize=(7,4))
    sns.regplot(data=d, x='age_mid', y='Num Reviews', scatter_kws={'s':8,'alpha':0.25})
    plt.yscale('log')
    plt.title('Age vs Num Reviews (log y)')
    plt.tight_layout()

if 'Average Rating' in d.columns:
    plt.figure(figsize=(7,4))
    sns.regplot(data=d, x='age_mid', y='Average Rating', scatter_kws={'s':8,'alpha':0.25})
    plt.title('Age vs Average Rating')
    plt.tight_layout()


## Category breakdown (top categories)

In [ ]:
if 'Categories' in d.columns:
    # Take the last breadcrumb after '>' as a simple leaf category
    leaf = d['Categories'].astype(str).str.split('>').str[-1].str.strip()
    vc = leaf.value_counts().head(15)
    plt.figure(figsize=(8,5))
    sns.barplot(x=vc.values, y=vc.index)
    plt.title('Top leaf Categories (count)')
    plt.tight_layout()
    vc